In [91]:
# Import Libraries
import pandas as pd
import numpy as np
import io
# from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import random

In [92]:
# ===================================================================
# Part 0: Helper Functions (Imputer and Evaluation)
# ===================================================================

def calculate_nan_euclidean(row1, row2):
    sum_sq_diff, valid_dimensions = 0.0, 0
    for i in range(len(row1)):
        if not pd.isna(row1.iloc[i]) and not pd.isna(row2.iloc[i]):
            sum_sq_diff += (row1.iloc[i] - row2.iloc[i])**2
            valid_dimensions += 1
    if valid_dimensions == 0: return np.inf
    return np.sqrt((len(row1) / valid_dimensions) * sum_sq_diff)

def knn_imputer_nan_euclidean(df, k=5):
    df_imputed = df.copy()
    rows_with_nan = df[df.isnull().any(axis=1)].index
    for row_index in rows_with_nan:
        row_to_impute = df.loc[row_index]
        missing_cols = row_to_impute[row_to_impute.isnull()].index.tolist()
        distances = []
        for i in df.index:
            if i != row_index:
                distances.append((i, calculate_nan_euclidean(row_to_impute, df.loc[i])))
        distances.sort(key=lambda x: x[1])
        neighbors = distances[:k]
        for col_to_impute in missing_cols:
            valid_neighbors = [(idx, dist) for idx, dist in neighbors if not pd.isna(df.loc[idx, col_to_impute])]
            if not valid_neighbors:
                imputed_value = df[col_to_impute].mean()
            else:
                weighted_sum, total_weight = 0.0, 0.0
                for neighbor_index, dist in valid_neighbors:
                    weight = 1 / (dist + 1e-6)
                    weighted_sum += df.loc[neighbor_index, col_to_impute] * weight
                    total_weight += weight
                imputed_value = weighted_sum / total_weight
            df_imputed.loc[row_index, col_to_impute] = imputed_value
    return df_imputed

In [93]:
class MyLogisticRegressionSGD:
    def __init__(self, learning_rate=0.01, n_epochs=100):
        self.learning_rate, self.n_epochs = learning_rate, n_epochs
        self.weights, self.bias = None, None

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        y_ = y.to_numpy()
        self.weights, self.bias = np.zeros(n_features), 0
        for _ in range(self.n_epochs):
            for i in range(n_samples):
                xi, yi = X[i], y_[i]
                linear_model = np.dot(xi, self.weights) + self.bias
                y_predicted = self._sigmoid(linear_model)
                dw, db = (y_predicted - yi) * xi, (y_predicted - yi)
                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db

    def predict(self, X, threshold=0.5):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted_proba = self._sigmoid(linear_model)
        return np.array([1 if i > threshold else 0 for i in y_predicted_proba])

def calculate_evaluation_metrics(y_true, y_pred):
    """
    Calculates evaluation metrics, prints them to the console,
    AND writes them to a file named 'model_output.txt'.
    """
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    accuracy = (TP + TN) / len(y_true)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    output_string = (
        "------------------------------------\n"
        "    Model Evaluation Metrics\n"
        "------------------------------------\n"
        f"Accuracy:  {accuracy:.4f}\n"
        f"Precision: {precision:.4f}\n"
        f"Recall:    {recall:.4f}\n"
        f"F1-Score:  {f1_score:.4f}\n"
        "------------------------------------\n"
    )

    print(output_string)

    with open("model_output.txt", "w") as f:
        f.write(output_string)

In [ ]:
# ===================================================================
# Main Script Execution
# ===================================================================

# --- STEP 1: LOAD AND PREPARE DATA ---
print("============================================")
print("     STEP 1: DATA LOADING & IMPUTATION")
print("============================================")
df = pd.read_csv('framingham.csv')
print(f"\nSuccessfully loaded")
print("\n--- Initial Missing Values Count ---")
print(df.isnull().sum())

# --- Pre-Imputation Cleaning Step ---
print("\n--- Pre-Processing: Dropping Rows ---")
missing_percent = df.isnull().mean() * 100
# Identify columns with >0% but <3% missing data
cols_to_clean = missing_percent[(missing_percent < 3) & (missing_percent > 0)].index
print(f"Found columns with minor missing data (<3%): {list(cols_to_clean)}")
df_cleaned = df.dropna(subset=cols_to_clean)
print(f"Original shape: {df.shape}")
print(f"Shape after dropping rows: {df_cleaned.shape}")

print("\n--- Missing Values Before KNN Imputation ---")
print(df_cleaned.isnull().sum())

# --- KNN Imputation Step ---
print("\nImputing remaining missing values using custom KNN imputer...")
df_imputed = knn_imputer_nan_euclidean(df_cleaned, k=5)
print("Imputation complete!")
print("\n--- Missing Values After Final Imputation ---")
print(df_imputed.isnull().sum())
if df_imputed.isnull().sum().sum() == 0:
    print("\nSuccess! No missing values remaining.")

     STEP 1: DATA LOADING & IMPUTATION

Successfully loaded

--- Initial Missing Values Count ---
male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

--- Pre-Processing: Dropping Rows ---
Found columns with minor missing data (<3%): ['education', 'cigsPerDay', 'BPMeds', 'totChol', 'BMI', 'heartRate']
Original shape: (4240, 16)
Shape after dropping rows: (3989, 16)

--- Missing Values Before KNN Imputation ---
male                 0
age                  0
education            0
currentSmoker        0
cigsPerDay           0
BPMeds               0
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol              0
sysBP                0
diaB

In [ ]:
# --- STEP 2: FEATURE SCALING AND DATA SPLITTING ---
print("\n\n============================================")
print("   STEP 2: FEATURE SCALING & DATA SPLITTING")
print("============================================")
X = df_imputed.drop('TenYearCHD', axis=1)
y = df_imputed['TenYearCHD']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data has been split into training (80%) and testing (20%) sets.")
print("Features have been scaled.")

In [ ]:
# --- STEP 3: MODEL TRAINING ---
print("\n\n============================================")
print("        STEP 3: MODEL TRAINING")
print("============================================")
print("Training Logistic Regression with Stochastic Gradient Descent (SGD)...")
log_reg_sgd = MyLogisticRegressionSGD(learning_rate=0.001, n_epochs=150)
log_reg_sgd.fit(X_train_scaled, y_train)
print("Model training complete!")

In [ ]:
# --- STEP 4: MODEL EVALUATION ---
print("\n\n============================================")
print("       STEP 4: MODEL EVALUATION")
print("============================================")
print("Evaluating the model's performance on the unseen test data...")
y_predictions = log_reg_sgd.predict(X_test_scaled)
calculate_evaluation_metrics(y_test, y_predictions)
print("Evaluation metrics saved to 'model_output.txt'")


# --- STEP 5: DISPLAY OUTPUT FILE ---
# print("\n\n============================================")
# print("     STEP 5: DISPLAYING OUTPUT FILE")
# print("============================================")
# print("Reading from 'model_output.txt'...\n")
# with open("model_output.txt", "r") as f:
#     print(f.read())

print("\n\n============================================")
print("             SCRIPT COMPLETE")
print("============================================")